# 03 - Value Implementation

这一节专门读 `micrograd.engine.Value` 的实现。

你已经知道：

```text
Value 可以参与 +、*、**、relu
L.backward() 后，每个 Value 会得到 grad
```

现在要搞清楚：**它内部是怎么做到的？**

本节目标：

1. 看懂 `Value.__init__` 保存了什么
2. 看懂 `__add__` 和 `__mul__` 如何创建新节点
3. 看懂 `_backward` 为什么是一个闭包
4. 看懂 `backward()` 为什么要先拓扑排序
5. 能用自己的话解释 `loss.backward()` 的最小原理


<!-- xiao-preview:start -->
## 课前预习作业：读源码前先预测局部规则

这节会读 `Value.__add__`、`Value.__mul__` 和 `Value.backward()`。先不看源码，预测两个局部反传规则。


In [1]:
# TODO: 把 None 改成你预测的结果。
# a + b 会创建一个新 Value，这个新节点的 _op 应该是什么？
preview_add_op = None

# out = a * b, a=2, b=3, out.grad=10
# 乘法局部反传时 a 和 b 各收到多少梯度？
preview_mul_to_a = None
preview_mul_to_b = None


def qa_check_03_preview(add_op, mul_to_a, mul_to_b):
    values = [add_op, mul_to_a, mul_to_b]
    if any(v is None for v in values):
        print('请先填写 TODO：先预测 _op 和乘法局部反传。')
        return
    assert add_op == '+', f"加法节点的 _op 应该是 '+'，但你填的是 {add_op!r}"
    assert mul_to_a == 30, f'a 收到的梯度应是 out.grad * b = 10 * 3 = 30，但你填的是 {mul_to_a}'
    assert mul_to_b == 20, f'b 收到的梯度应是 out.grad * a = 10 * 2 = 20，但你填的是 {mul_to_b}'
    print('OK: 预习通过，现在可以读源码。')


qa_check_03_preview(preview_add_op, preview_mul_to_a, preview_mul_to_b)


请先填写 TODO：先预测 _op 和乘法局部反传。


<!-- xiao-preview:hint -->
<details>
<summary><strong>Show / Hide 提示</strong></summary>

`_op` 用来记录这个节点是怎么来的。乘法反传时，梯度要乘上另一个输入的数值。

</details>


<!-- xiao-preview:answer -->
<details>
<summary><strong>Show / Hide 答案</strong></summary>

```python
preview_add_op = '+'
preview_mul_to_a = 30
preview_mul_to_b = 20
```

</details>


## 练习方式：先读源码，再补小块

这一节不是让你照着源码抄一遍，而是按这个顺序练：

```text
1. 先观察真实源码做了什么
2. 用小数字把 self / other / out 代进去
3. 在留空练习里补 1-3 行关键代码
4. 运行 qa_check 看是否理解对
5. 卡住时先看提示，最后才看答案
```

重点不是背代码，而是搞清楚：每个运算在前向时保存了什么，反向时怎么把 `out.grad` 传给父节点。

## 本节路线图：四次接触同一个思想

同一个概念会出现四次，难度一点点加：

| 次数 | 你做什么 | 目标 |
|---|---|---|
| 1 | 看 micrograd 真源码 | 知道真实实现长什么样 |
| 2 | 把 `a+b` 代入 `self/other/out` | 消除 Python 语法障碍 |
| 3 | 做留空练习并运行 `qa_check` | 确认自己能补关键规则 |
| 4 | 写一个小版本并测试 | 从“看懂”变成“能写” |

这才是这一节的学习方式：源码给直觉，练习给掌握。

## 0. Setup

导入本地源码里的 `Value`。这本 notebook 主要读源码和做小实验。


In [2]:
from pathlib import Path
import sys
import inspect

cwd = Path.cwd().resolve()
candidates = [cwd, cwd.parent, cwd / 'micrograd', cwd.parent / 'micrograd']
PROJECT_ROOT = None
for candidate in candidates:
    if (candidate / 'micrograd' / 'engine.py').exists():
        PROJECT_ROOT = candidate
        break

if PROJECT_ROOT is None:
    raise RuntimeError(f'Could not find local micrograd package from {cwd}')

sys.path.insert(0, str(PROJECT_ROOT))

from micrograd.engine import Value
print('project root:', PROJECT_ROOT)



def _all_zero(*values):
    return all(abs(value) < 1e-12 for value in values)


def qa_check_add_backward_rule(Cls):
    a = Cls(2.0)
    b = Cls(3.0)
    c = a + b
    c.grad = 5.0
    c._backward()
    if _all_zero(a.grad, b.grad):
        print('TODO: 请先补全加法的 _backward。')
        return
    assert a.grad == 5.0, f'a.grad expected 5.0, got {a.grad}'
    assert b.grad == 5.0, f'b.grad expected 5.0, got {b.grad}'
    print('OK: add backward rule passed.')


def qa_check_mul_backward_rule(Cls):
    a = Cls(2.0)
    b = Cls(3.0)
    c = a * b
    c.grad = 5.0
    c._backward()
    if _all_zero(a.grad, b.grad):
        print('TODO: 请先补全乘法的 _backward。')
        return
    assert a.grad == 15.0, f'a.grad expected 15.0, got {a.grad}'
    assert b.grad == 10.0, f'b.grad expected 10.0, got {b.grad}'
    print('OK: mul backward rule passed.')


def qa_check_topo_order(order):
    if order is None:
        print('TODO: 请先填写一个合法的 topo 顺序列表。')
        return
    required = [('a', 'c'), ('b', 'c'), ('c', 'd'), ('a', 'd'), ('d', 'L'), ('const2', 'L')]
    pos = {name: i for i, name in enumerate(order)}
    missing = {name for pair in required for name in pair if name not in pos}
    assert not missing, f'missing nodes: {sorted(missing)}'
    for parent, child in required:
        assert pos[parent] < pos[child], f'{parent} must appear before {child}'
    print('OK: topo order is valid.')


project root: /Users/barry/IdeaProjects/llm/micrograd


## 1. Value Is A Number Plus Autograd Metadata

先看构造函数：


In [3]:
print(inspect.getsource(Value.__init__))


    def __init__(self, data, _children=(), _op=''):
        self.data = data
        self.grad = 0
        # internal variables used for autograd graph construction
        self._backward = lambda: None
        self._prev = set(_children)
        self._op = _op # the op that produced this node, for graphviz / debugging / etc



`Value` 里最重要的字段：

```text
data       当前节点的数值
       grad       最终输出对当前节点的导数/偏导，初始是 0
_backward  一个函数，负责把当前节点的 grad 传给父节点
_prev      父节点集合，也就是当前节点从哪些节点算出来
_op        当前节点由什么运算产生，比如 +、*、ReLU
```

叶子节点最简单：


In [4]:
a = Value(2.0)
print('a.data =', a.data)
print('a.grad =', a.grad)
print('a._prev =', a._prev)
print('a._op =', repr(a._op))
print('a._backward =', a._backward)


a.data = 2.0
a.grad = 0
a._prev = set()
a._op = ''
a._backward = <function Value.__init__.<locals>.<lambda> at 0x107e5df80>


## 2. Addition Creates A New Value Node

看 `__add__`：


In [5]:
print(inspect.getsource(Value.__add__))


    def __add__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        out = Value(self.data + other.data, (self, other), '+')

        def _backward():
            self.grad += out.grad
            other.grad += out.grad
        out._backward = _backward

        return out



##### 这段代码做了两件事：

```text
1. 前向：out.data = self.data + other.data
2. 反向规则：如果 out 收到梯度 out.grad，就把它原样传给 self 和 other
```

因为：

$$
out = x + y
$$

所以：

$$
\frac{\partial out}{\partial x}=1, \quad \frac{\partial out}{\partial y}=1
$$

也就是：

```python
self.grad += out.grad
other.grad += out.grad
```

做个小实验：


In [6]:
a = Value(2.0)
b = Value(3.0)
c = a + b

print('c =', c)
print('c._op =', c._op)
print('c._prev =', c._prev)

print('parents data =', [v.data for v in c._prev])

c.backward()
print('a.grad =', a.grad)
print('b.grad =', b.grad)


c = Value(data=5.0, grad=0)
c._op = +
c._prev = {Value(data=2.0, grad=0), Value(data=3.0, grad=0)}
parents data = [2.0, 3.0]
a.grad = 1
b.grad = 1


## 2.1 用具体例子理解 `__add__`

如果你对 Python 不太熟，先不用管“魔法方法”这个名字。只记住：

```python
c = a + b
```

当 `a` 是 `Value` 时，Python 会自动调用：

```python
a.__add__(b)
```

所以在源码里：

```python
def __add__(self, other):
```

对应关系就是：

```text
self  = a
other = b
```

如果：

```python
a = Value(2.0)
b = Value(3.0)
c = a + b
```

那么可以把 `__add__` 临时翻译成下面这个更直白的版本：

```python
def add(a, b):
    b = b if isinstance(b, Value) else Value(b)

    c = Value(a.data + b.data, (a, b), '+')

    def _backward():
        a.grad += c.grad
        b.grad += c.grad

    c._backward = _backward
    return c
```

这段做了三件事：

```text
1. 算前向结果：c.data = a.data + b.data = 5
2. 记录来源：c._prev = {a, b}, c._op = '+'
3. 绑定反向规则：以后 c 收到 grad，就把它传回 a 和 b
```


### Python 补给站：`__add__` 为什么会被调用

Python 看到：

```python
c = a + b
```

如果 `a` 是 `Value`，它会尝试调用：

```python
a.__add__(b)
```

所以读源码时永远可以这样代入：

```text
self  = 加号左边的 Value，也就是 a
other = 加号右边的值，也就是 b
out   = 这次加法新创建出来的结果节点 c
```

这个代入法也适用于乘法：

```text
c = a * b  等价于  a.__mul__(b)
```

In [7]:
a = Value(2.0)
b = Value(3.0)
c = a + b

print('a =', a)
print('b =', b)
print('c = a + b =', c)
print()
print('c.data =', c.data)
print('c.grad =', c.grad)
print('c._op =', c._op)
print('c._prev data =', [parent.data for parent in c._prev])
print('c._backward =', c._backward)


a = Value(data=2.0, grad=0)
b = Value(data=3.0, grad=0)
c = a + b = Value(data=5.0, grad=0)

c.data = 5.0
c.grad = 0
c._op = +
c._prev data = [3.0, 2.0]
c._backward = <function Value.__add__.<locals>._backward at 0x107e5f2e0>


## 2.2 手动调用加法节点的 `_backward`

正常情况下，我们调用：

```python
c.backward()
```

它会自动设置：

```python
c.grad = 1
```

然后再调用：

```python
c._backward()
```

为了看清楚，我们这里手动做一次：

```text
c = a + b
加法局部导数：dc/da = 1, dc/db = 1
所以：a.grad += c.grad, b.grad += c.grad
```


In [8]:
a = Value(2.0)
b = Value(3.0)
c = a + b

print('before manual backward')
print('a.grad =', a.grad)
print('b.grad =', b.grad)
print('c.grad =', c.grad)

# Pretend c is the final output. The final output's gradient to itself is 1.
c.grad = 1
c._backward()

print('\nafter c.grad = 1 and c._backward()')
print('a.grad =', a.grad)
print('b.grad =', b.grad)
print('c.grad =', c.grad)


before manual backward
a.grad = 0
b.grad = 0
c.grad = 0

after c.grad = 1 and c._backward()
a.grad = 1
b.grad = 1
c.grad = 1


## 2.3 为什么要处理 `a + 3`

源码第一行：

```python
other = other if isinstance(other, Value) else Value(other)
```

是为了让下面这种写法也能工作：

```python
a = Value(2.0)
c = a + 3
```

因为普通数字 `3` 没有 `.data`、`.grad`、`._backward` 这些属性，所以要先包装成：

```python
Value(3)
```

这样加法逻辑就统一了：

```text
Value + Value
```


In [9]:
a = Value(2.0)
c = a + 3

print('c =', c)
print('c.data =', c.data)
print('parents data =', [parent.data for parent in c._prev])

c.backward()
print('a.grad =', a.grad)


c = Value(data=5.0, grad=0)
c.data = 5.0
parents data = [3, 2.0]
a.grad = 1


## 3. Multiplication Creates A New Value Node

看 `__mul__`：


In [10]:
print(inspect.getsource(Value.__mul__))


    def __mul__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        out = Value(self.data * other.data, (self, other), '*')

        def _backward():
            self.grad += other.data * out.grad
            other.grad += self.data * out.grad
        out._backward = _backward

        return out



乘法的局部导数是：

$$
out = xy
$$

$$
\frac{\partial out}{\partial x}=y, \quad \frac{\partial out}{\partial y}=x
$$

所以 micrograd 写成：

```python
self.grad += other.data * out.grad
other.grad += self.data * out.grad
```

这里的 `out.grad` 是链式法则里“从后面传来的梯度”。

小实验：


In [11]:
a = Value(2.0)
b = Value(3.0)
c = a * b

c.backward()
print('c.data =', c.data)
print('a.grad =', a.grad)  # dc/da = b = 3
print('b.grad =', b.grad)  # dc/db = a = 2


c.data = 6.0
a.grad = 3.0
b.grad = 2.0


## 4. Why `_backward` Is A Closure

`_backward` 是定义在 `__add__` / `__mul__` 里面的函数。

它能记住当时参与运算的变量：

```python
self
other
out
```

这叫闭包。直观理解：

```text
前向计算时，把“将来怎么反传”这张小纸条贴到 out 节点上。
等 backward 真正发生时，再读这张纸条，把梯度传回父节点。
```

下面手动调用 `_backward`，看它确实会修改父节点的 grad：


In [12]:
a = Value(2.0)
b = Value(3.0)
c = a * b

# Normally backward() sets the final output grad to 1.
# Here we do it manually to inspect the local backward rule.
c.grad = 1
c._backward()

print('after c._backward():')
print('a.grad =', a.grad)
print('b.grad =', b.grad)


after c._backward():
a.grad = 3.0
b.grad = 2.0


## 5. Why Use `+=` Instead Of `=`

因为一个变量可能通过多条路径影响最终输出。

例如：

$$
L = a + a
$$

数学上：

$$
\frac{dL}{da}=2
$$

如果反向传播时不用 `+=` 累加，就会丢掉其中一条路径的贡献。


In [13]:
a = Value(2.0)
L = a + a
L.backward()

print('L.data =', L.data)
print('a.grad =', a.grad)


L.data = 4.0
a.grad = 2


## 6. Backward Needs Topological Order

现在看 `backward()` 源码：


In [14]:
print(inspect.getsource(Value.backward))


    def backward(self):

        # topological order all of the children in the graph
        topo = []
        visited = set()
        def build_topo(v):
            if v not in visited:
                visited.add(v)
                for child in v._prev:
                    build_topo(child)
                topo.append(v)
        build_topo(self)

        # go one variable at a time and apply the chain rule to get its gradient
        self.grad = 1
        for v in reversed(topo):
            v._backward()



为什么要拓扑排序？

因为反向传播时，必须保证：

```text
先处理最终输出
再处理它的父节点
再处理父节点的父节点
```

如果顺序乱了，某个节点可能还没收到完整梯度就开始往前传。

`topo` 是从叶子到输出的顺序，`reversed(topo)` 才是反向传播执行顺序。


In [15]:
def build_demo_graph():
    a = Value(2.0)
    b = Value(3.0)
    c = a * b
    d = c + a
    L = d * 2
    return {'a': a, 'b': b, 'c': c, 'd': d, 'L': L}

nodes = build_demo_graph()
L = nodes['L']

# Rebuild the topological order, just like Value.backward does.
topo = []
visited = set()

def build_topo(v):
    if v not in visited:
        visited.add(v)
        for child in v._prev:
            build_topo(child)
        topo.append(v)

build_topo(L)

def name_of(value):
    for name, node in nodes.items():
        if node is value:
            return name
    return f'Value({value.data})'

print('topo:', ' -> '.join(name_of(v) for v in topo))
print('backward order:', ' -> '.join(name_of(v) for v in reversed(topo)))


topo: Value(2) -> a -> b -> c -> d -> L
backward order: L -> d -> c -> b -> a -> Value(2)


## 6.1 `_prev` 是 set，所以 topo 顺序不是唯一的

在 `Value.__init__` 里有这一行：

```python
self._prev = set(_children)
```

`set` 的特点是：

```text
1. 会去重
2. 不保证遍历顺序
```

所以对于：

```python
c = a * b
```

逻辑上：

```text
c._prev = {a, b}
```

但是执行：

```python
for child in c._prev:
```

可能先看到 `a`，也可能先看到 `b`。这不是按变量名字母序，也不是按 `data` 大小排序。

因此 `topo` 的输出不是唯一答案。重点不是：

```text
topo 必须等于 a -> b -> c -> d -> Value(2) -> L
```

而是：

```text
父节点必须出现在子节点前面
```

也就是：

```text
a、b 在 c 前面
c、a 在 d 前面
d、Value(2) 在 L 前面
```

只要满足这个依赖关系，就是合法的拓扑顺序。


In [16]:
nodes = build_demo_graph()
L = nodes['L']

topo = []
visited = set()

def build_topo(v):
    if v not in visited:
        visited.add(v)
        for child in v._prev:
            build_topo(child)
        topo.append(v)

build_topo(L)

# Give the anonymous constant Value(2) a readable name for this check.
def stable_name(value):
    for name, node in nodes.items():
        if node is value:
            return name
    return f'const({value.data})'

names = [stable_name(v) for v in topo]
position = {v: i for i, v in enumerate(topo)}

print('one valid topo order:')
print(' -> '.join(names))
print()
print('dependency checks:')
for child in topo:
    print('-- child', stable_name(child), child._prev)
    for parent in child._prev:
        print(f'{stable_name(parent):>8} before {stable_name(child):>8}:', position[parent] < position[child])


one valid topo order:
const(2) -> a -> b -> c -> d -> L

dependency checks:
-- child const(2) set()
-- child a set()
-- child b set()
-- child c {Value(data=2.0, grad=0), Value(data=3.0, grad=0)}
       a before        c: True
       b before        c: True
-- child d {Value(data=6.0, grad=0), Value(data=2.0, grad=0)}
       c before        d: True
       a before        d: True
-- child L {Value(data=2, grad=0), Value(data=8.0, grad=0)}
const(2) before        L: True
       d before        L: True


如果你看到 `topo` 顺序变了，不要紧张。

比如下面这些都可能合法：

```text
a -> b -> c -> d -> const(2) -> L
b -> a -> c -> d -> const(2) -> L
const(2) -> a -> b -> c -> d -> L
```

只要父节点在子节点前面，反向传播就能正常工作。

`reversed(topo)` 用来反向执行时，也不要求唯一顺序。对同一层互不依赖的节点，谁先谁后通常不影响最终梯度。


## 7. Trace One Backward Pass

我们用同一个例子：

$$
L = 2(ab+a)
$$

前向：

```text
a=2, b=3, c=6, d=8, L=16
```

反向后应该得到：

```text
a.grad = 8
b.grad = 4
```


In [17]:
nodes = build_demo_graph()

def show(nodes, title):
    print(title)
    for name in ['a', 'b', 'c', 'd', 'L']:
        v = nodes[name]
        parents = []
        for p in v._prev:
            parents.append(name_of_node(p, nodes))
        print(f'{name:>2} | data={v.data:>5} | grad={v.grad:>5} | op={v._op or "leaf":>4} | parents={parents}')

def name_of_node(value, nodes):
    for name, node in nodes.items():
        if node is value:
            return name
    return f'Value({value.data})'

print(nodes)
show(nodes, 'before backward')
nodes['L'].backward()
print()
show(nodes, 'after backward')


{'a': Value(data=2.0, grad=0), 'b': Value(data=3.0, grad=0), 'c': Value(data=6.0, grad=0), 'd': Value(data=8.0, grad=0), 'L': Value(data=16.0, grad=0)}
before backward
 a | data=  2.0 | grad=    0 | op=leaf | parents=[]
 b | data=  3.0 | grad=    0 | op=leaf | parents=[]
 c | data=  6.0 | grad=    0 | op=   * | parents=['a', 'b']
 d | data=  8.0 | grad=    0 | op=   + | parents=['a', 'c']
 L | data= 16.0 | grad=    0 | op=   * | parents=['d', 'Value(2)']

after backward
 a | data=  2.0 | grad=  8.0 | op=leaf | parents=[]
 b | data=  3.0 | grad=  4.0 | op=leaf | parents=[]
 c | data=  6.0 | grad=    2 | op=   * | parents=['a', 'b']
 d | data=  8.0 | grad=    2 | op=   + | parents=['a', 'c']
 L | data= 16.0 | grad=    1 | op=   * | parents=['d', 'Value(2)']


## TODO Lab - 局部反传出口检查

下面这些练习不要求你重写完整 `Value`，只补最关键的小块：

```text
加法怎么把 out.grad 传回输入
乘法为什么要乘另一个输入的 data
topo 顺序为什么保证先有下游 grad
进入 04 前能不能讲清楚这些点
```

这一节还是源码理解课；完整实现留到 `04_tinyvalue_exercises.ipynb`。

## 8. Exercise - 补全加法局部反传

这一题只练 `__add__` 里的局部反向规则。

先不要看答案。把 `_backward` 里的 `pass` 改掉，然后运行 `qa_check_add_backward_rule`。

In [18]:
class AddRuleExercise:
    def __init__(self, data, children=(), op=''):
        self.data = data
        self.grad = 0
        self._prev = set(children)
        self._op = op
        self._backward = lambda: None

    def __add__(self, other):
        other = other if isinstance(other, AddRuleExercise) else AddRuleExercise(other)
        out = AddRuleExercise(self.data + other.data, (self, other), '+')

        def _backward():
            # TODO: out = self + other，补全梯度怎么传给 self 和 other
            pass

        out._backward = _backward
        return out


qa_check_add_backward_rule(AddRuleExercise)

TODO: 请先补全加法的 _backward。


<details>
<summary><strong>Show / Hide 提示</strong></summary>

加法 `out = self + other`，两个局部导数都是 1。所以从后面传来的 `out.grad` 要原样加到两个父节点上。

</details>

<details>
<summary><strong>Show / Hide 答案</strong></summary>

```python
def _backward():
    self.grad += out.grad
    other.grad += out.grad
```

</details>

## 9. Exercise - 补全乘法局部反传

这一题只练 `__mul__` 里的局部反向规则。

In [19]:
class MulRuleExercise:
    def __init__(self, data, children=(), op=''):
        self.data = data
        self.grad = 0
        self._prev = set(children)
        self._op = op
        self._backward = lambda: None

    def __mul__(self, other):
        other = other if isinstance(other, MulRuleExercise) else MulRuleExercise(other)
        out = MulRuleExercise(self.data * other.data, (self, other), '*')

        def _backward():
            # TODO: out = self * other，补全梯度怎么传给 self 和 other
            pass

        out._backward = _backward
        return out


qa_check_mul_backward_rule(MulRuleExercise)

TODO: 请先补全乘法的 _backward。


<details>
<summary><strong>Show / Hide 提示</strong></summary>

`out = self * other`。对 `self` 的局部导数是 `other.data`；对 `other` 的局部导数是 `self.data`。还要乘上后面传来的 `out.grad`。

</details>

<details>
<summary><strong>Show / Hide 答案</strong></summary>

```python
def _backward():
    self.grad += other.data * out.grad
    other.grad += self.data * out.grad
```

</details>

## 10. Exercise - 判断合法 topo 顺序

这一题不写类，只检查你是否理解 topo。

规则只有一句话：

```text
父节点必须在子节点前面。
```

In [20]:
# 这个图来自：
# c = a * b
# d = c + a
# L = d * const2
#
# TODO: 填一个合法的 topo 顺序。
# 规则：父节点必须出现在子节点前面。
# 可用节点名：'a', 'b', 'c', 'd', 'const2', 'L'

student_topo_order = None

qa_check_topo_order(student_topo_order)

TODO: 请先填写一个合法的 topo 顺序列表。


<details>
<summary><strong>Show / Hide 提示</strong></summary>

`c` 依赖 `a,b`；`d` 依赖 `c,a`；`L` 依赖 `d,const2`。因此 `a,b` 必须在 `c` 前面，`c` 必须在 `d` 前面，`d,const2` 必须在 `L` 前面。

</details>

<details>
<summary><strong>Show / Hide 答案</strong></summary>

一种合法答案：

```python
student_topo_order = ['a', 'b', 'c', 'd', 'const2', 'L']
```

注意：这不是唯一答案。`b` 可以在 `a` 前面，`const2` 也可以更早出现。

</details>

## 11. Exit Check - 进入 04 前先确认

这一节不要求你写完整 `Value`。

你只需要确认自己能回答下面 4 个问题；完整实现放到下一节 `04_tinyvalue_exercises.ipynb`。

In [21]:
# TODO: 把 None 改成你的简短回答字符串。
# 这不是自动判分题，而是帮助你确认有没有准备好进入 04。

answers = {
    'add_backward_rule': None,      # 例：加法的 out.grad 怎么传给父节点？
    'mul_backward_rule': None,      # 例：乘法为什么要乘 other.data / self.data？
    'why_accumulate_grad': None,    # 例：为什么用 += 而不是 =？
    'why_topo': None,               # 例：为什么 backward 前要先 topo？
}


def qa_check_exit_answers(answers):
    missing = [key for key, value in answers.items() if value is None or str(value).strip() == '']
    if missing:
        print('TODO: 还有这些问题没有回答：', missing)
        return
    print('OK: 你已经写下自己的理解。现在可以进入 04 做完整实现练习。')


qa_check_exit_answers(answers)

TODO: 还有这些问题没有回答： ['add_backward_rule', 'mul_backward_rule', 'why_accumulate_grad', 'why_topo']


<details>
<summary><strong>Show / Hide 提示</strong></summary>

可以用很短的话回答，不需要写成论文：

```text
add_backward_rule：加法局部导数都是 1，所以 out.grad 原样给两边。
mul_backward_rule：out=x*y，对 x 的导数是 y，对 y 的导数是 x。
why_accumulate_grad：同一个变量可能有多条路径影响 L，贡献要相加。
why_topo：必须等一个节点收齐来自后面的梯度，再继续往它的父节点传。
```

</details>

<details>
<summary><strong>Show / Hide 答案</strong></summary>

一种可接受的回答：

```python
answers = {
    'add_backward_rule': 'out = x + y，所以 dx 和 dy 都收到 out.grad',
    'mul_backward_rule': 'out = x * y，所以 x.grad 加 y.data*out.grad，y.grad 加 x.data*out.grad',
    'why_accumulate_grad': '一个变量可能从多条路径影响 L，所以梯度贡献要 += 累加',
    'why_topo': '反向传播要按依赖反过来走，确保每个节点先收到完整梯度再往前传',
}
```

</details>

## 12. Debug Lab

下面这些是调试练习，不需要现在全做：

```text
1. 在 AddRuleExercise.__add__ 的 _backward 里，把 += 改成 =，再跑 L = a + a。
   观察 a.grad 为什么可能丢掉一条路径。

2. 在 build_topo 里删掉 visited 判断。
   观察重复节点会怎样进入 topo。

3. 在 __mul__ 的 _backward 里故意把 self/other 写反。
   观察 a.grad 和 b.grad 如何交换。
```

学源码最有效的方法之一，就是制造一个小 bug，然后看测试怎么抓住它。

## 13. What To Remember

这一节过关，只需要能讲清楚这 5 句话：

```text
1. Value 保存 data，也保存 grad 和计算图信息
2. 每次 +、* 都会创建一个新的 Value 节点
3. 新节点会保存父节点 _prev 和运算符 _op
4. 新节点还会保存一个 _backward 函数，记录局部反传规则
5. backward() 先拓扑排序，再倒序调用每个节点的 _backward
```

下一步才适合进入：

```text
04_tinyvalue_exercises.ipynb
```

在那里你会写更完整的 TinyValue，包括 `**` 和 `relu`。

<!-- xiao-post:start -->
## 课后作业提交清单

这一节学完后，用下面 5 条自检：

```text
1. 我能把 c = a + b 代入 self=a, other=b, out=c 讲清楚。
2. 我能解释 + 和 * 为什么都会创建新 Value 节点。
3. 我能解释局部 _backward 在做什么。
4. 我能解释为什么 grad 用 +=，为什么 L.grad 先设成 1。
5. 我能解释 topo 的目的，但还不需要从零写完整 Value。
```


## AI 复盘检查 Prompt

把下面这段发送给任意 AI，让它检查你是否理解 `Value` 的实现：

```text
你是我的 micrograd Value 源码实现检查官。

我刚学完一节内容，主题是：micrograd.engine.Value 如何实现前向建图和反向传播。

请你用一问一答的方式检查我。规则：
1. 一次只问一个问题，等我回答后再评价。
2. 不要直接给完整答案，除非我连续两次答不出来。
3. 如果我混淆了 Python 语法，请先用具体例子解释，比如 c = a + b 时 self、other、out 分别是谁。
4. 每个问题都要追问一句“你能举一个具体例子吗？”
5. 最后给我一个 0-100 的掌握度评分，并判断我是否可以进入 nn.py。

请按这个顺序检查我：
1. Value.__init__ 里 data、grad、_prev、_op、_backward 分别保存什么？
2. 执行 c = a + b 时，Python 为什么会调用 a.__add__(b)？此时 self 和 other 分别是谁？
3. __add__ 里为什么要把普通数字包装成 Value？例如 a + 3。
4. __add__ 创建 out = Value(...) 时，out.data、out._prev、out._op 分别是什么？
5. 加法节点的 _backward 做什么？为什么是 self.grad += out.grad，other.grad += out.grad？
6. 乘法节点的 _backward 做什么？为什么要乘 other.data 或 self.data？
7. _backward 为什么说是闭包？它记住了哪些变量？
8. 为什么 grad 要用 +=，而不是 =？请用 L = a + a 举例。
9. 为什么 backward() 一开始要把最终节点的 grad 设成 1？
10. backward() 为什么要先做拓扑排序，再反向调用每个节点的 _backward？

现在从第 1 个问题开始问我。
```
